In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

df = pd.read_csv(r"C:/Users/USA/3mtt/3mmt-project/data_2.0.csv")
print(df.shape)
display(df.head())


Exploring the data set to find missing values, duplicates to ensure data quality before analysis

In [ ]:
# Basic information
print("=== Dataset Information ===")
df.info()

print("\n=== Summary Statistics ===")
display(df.describe())

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Number of Duplicates ===")
print(df.duplicated().sum())

# Handle missing values if any
df = df.dropna()
print("\nData cleaned. New shape:", df.shape)

Visualizing the relationship between marketing spend on TV, Radio and Social media vs sales. To understand the relationships between them.

In [ ]:
# Pairplot with regression lines
sns.pairplot(df, x_vars=['TV', 'Radio', 'Social_Media'], y_vars='Sales', 
             height=5, aspect=0.8, kind='reg', plot_kws={'line_kws':{'color': 'blue'}})
plt.suptitle('Marketing Channels vs Sales', y=1.02)
plt.show()

# Correlation heatmap
plt.figure(figsize=(8, 6))
corr = df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.3f')
plt.title('Correlation Matrix')
plt.show()

# Individual scatter plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(['TV', 'Radio', 'Social_Media']):
    sns.regplot(x=col, y='Sales', data=df, ax=axes[i], line_kws={'color': 'blue'})
    axes[i].set_title(f'{col} vs Sales')
plt.tight_layout()
plt.show()

Identifying the Market spend(TV,radio,Social_Media) with the most linear relationship with Sales.

In [ ]:
correlations = df[['TV', 'Radio', 'Social_Media', 'Sales']].corr()['Sales'].sort_values(ascending=False)
print("Correlation with Sales:\n")
print(correlations)

# Select best predictor
best_predictor = correlations.index[1] # Skip 'Sales'
print(f"\n Best independent variable for prediction: {best_predictor}") 

OLS regression Results:
Sales = (3.5615 X TV) - 0.1325$$ just like y = mx + c

In [ ]:
# Prepare variables
X = df[[best_predictor]]
y = df['Sales']

# Add constant (intercept)
X = sm.add_constant(X)

# Fit the model
model = sm.OLS(y, X).fit()

# Display results
print(model.summary())

In [ ]:
fitted = model.fittedvalues
residuals = model.resid

# Residuals vs Fitted
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.scatterplot(x=fitted, y=residuals)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values')

# Q-Q Plot
plt.subplot(1, 2, 2)
sm.qqplot(residuals, line='45', fit=True)
plt.title('Q-Q Plot')

plt.tight_layout()
plt.show()

# Residual distribution
plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True)
plt.title('Distribution of Residuals')
plt.show()

 

The statistical outputs (R-squared, coefficients, p-values) are translated into actionable marketing insights.

In [ ]:
print("=== FINAL MODEL INTERPRETATION ===\n")

print(f"• R-squared: {model.rsquared:.4f} → The model explains {model.rsquared*100:.1f}% of the variation in Sales.")

coef = model.params[best_predictor]
pvalue = model.pvalues[best_predictor]

print(f"• For every $1 increase in **{best_predictor}** spending, Sales increase by **${coef:.2f}** on average.")

if pvalue < 0.05:
    print(f"• The result is statistically significant (p-value = {pvalue:.4f})")
else:
    print(f"• The result is NOT statistically significant (p-value = {pvalue:.4f})")

print("\n=== BUSINESS RECOMMENDATION ===")
print(f"**Allocate more marketing budget to {best_predictor}** as it demonstrates the strongest positive impact on Sales.")
print("This channel offers the highest ROI and should be prioritized in future campaigns.")